# Twinkle 数学 GRPO 训练

> 🎯 **训练目标**：通过 GRPO 强化学习，让模型在解数学题时 **减少 thinking 长度**，用更简短的推理过程得出正确答案。

本 Notebook 演示如何使用 **GRPO（Group Relative Policy Optimization）** 算法在数学问题上训练语言模型。

### 什么是 GRPO？

GRPO 是一种强化学习算法，训练流程如下：
1. 对每个数学题，让模型 **生成多个回答**
2. 用奖励函数 **评分** 每个回答（答案是否正确、格式是否规范）
3. 在同组内 **计算优势值**（advantage）：好回答得正值，差回答得负值
4. 用优势值 **更新策略**：让模型更倾向于生成好答案

### 整体流程

```
准备数据集 → 初始化训练/采样客户端 → 训练循环：
  同步权重 → 采样生成 → 计算奖励 → 计算优势 → GRPO 训练 → 日志记录
```

### 前置条件

| 条件 | 说明 |
|------|------|
| 环境变量 | 设置 `MODELSCOPE_TOKEN` |
| 依赖安装 | `pip install twinkle-kit[tinker]` |

> 💡 **获取 Token**：访问 [ModelScope Token 页面](https://www.modelscope.cn/my/access/token) 获取你的 `MODELSCOPE_TOKEN`，并设置为环境变量：`export MODELSCOPE_TOKEN=<你的Token>`


## 🚀 零卡训练服务化（Serverless Training）

本 Notebook 运行在 **ModelScope 零卡训练平台** 上。你无需自备 GPU，只需在 Notebook 中编写训练逻辑，平台会自动调度云端 GPU 资源完成训练。

### 架构示意图

```
┌─────────────────────────────────────────────────────────────┐
│                    你的 Notebook（CPU 环境）                   │
│                                                             │
│   ┌──────────┐    HTTP / gRPC     ┌──────────────────────┐  │
│   │  Twinkle │ ─────────────────► │   ModelScope 云端     │  │
│   │  Client  │ ◄───────────────── │   GPU 训练集群        │  │
│   └──────────┘    训练结果返回      │                      │  │
│       │                           │  ┌────┐ ┌────┐ ┌────┐│  │
│       │  构造数据                   │  │GPU0│ │GPU1│ │... ││  │
│       │  发送训练请求               │  └────┘ └────┘ └────┘│  │
│       │  接收指标/检查点            │  模型加载 + LoRA 训练  │  │
│       ▼                           └──────────────────────┘  │
│   ┌──────────┐                                              │
│   │ 数据准备  │  Dataset / DataLoader / Preprocessor         │
│   └──────────┘                                              │
└─────────────────────────────────────────────────────────────┘
```

### 核心优势

| 特性 | 说明 |
|------|------|
| **零卡启动** | Notebook 本身不需要 GPU，训练在云端自动执行 |
| **按需付费** | 仅在训练时占用 GPU 资源 |
| **开箱即用** | 预置主流模型，无需下载权重 |
| **LoRA 微调** | 高效参数微调，几分钟即可完成小规模训练 |

> 🔗 本项目由 [Twinkle](https://github.com/modelscope/twinkle) 框架提供支持 | [GitHub](https://github.com/modelscope/twinkle)

## 第一步：导入依赖与全局配置

> **为什么使用 Twinkle 客户端语法？**
> Twinkle 提供 `tinker` 和 `twinkle` 两套客户端 API。其中 **tinker** 接口不支持设置 `target_modules`、`LoraConfig` 等细节调控，而 GRPO 训练在 MoE 模型上需要显式指定 LoRA 的 target modules（否则会触发 vLLM 兼容性问题）。
> 因此本 Notebook 使用 **twinkle 客户端语法**，以获得对训练参数的完整控制。

| 配置项 | 默认值 | 说明 |
|--------|--------|------|
| `MODEL_ID` | ms://Qwen/Qwen3.6-35B-A3B | 基座模型（需加 `ms://` 前缀） |
| `NUM_GENERATIONS` | 4 | 每个 prompt 生成几条回答 |
| `MAX_NEW_TOKENS` | 1024 | 单条回答最大 token 数 |
| `LEARNING_RATE` | 2e-5 | 学习率 |
| `MAX_STEPS` | 100 | 最大训练步数 |
| `BATCH_SIZE` | 2 | 每步的 prompt 数量（实际训练样本 = BATCH_SIZE × NUM_GENERATIONS） |
| `TEMPERATURE` | 1.0 | 采样温度，RL 训练中通常设为 1.0 保持多样性 |
| `SYNC_INTERVAL` | 1 | 每隔多少步同步权重到采样端 |

### ⚠️ MoE 模型 LoRA 注意事项

由于 `Qwen/Qwen3.6-35B-A3B` 是 MoE（Mixture of Experts）架构，在配合 vLLM 采样时存在已知兼容性问题。
如果你在本地使用 Megatron 进行 GRPO 训练，建议显式指定 `target_modules`（而非 `all-linear`）：

```python
target_modules:
    - mlp.linear_fc1
    - mlp.linear_fc2
    - attn.proj
    - shared_experts.linear_fc1
    - shared_experts.linear_fc2
    - linear_qkv
    - in_proj
    - out_proj
    - linear_proj
```

> **注意**：此配置是一个示例，由于问题来自 vLLM 侧的 MoE LoRA 支持尚不完善，实际训练效果可能受限。
> 如果不需要在线采样（vLLM），使用 `all-linear` 仍然可以正常训练。

## 环境安装

首次运行前，请先执行以下安装命令。如已安装可跳过此步。

In [1]:
!pip install twinkle-kit[tinker] -q

In [2]:
import dotenv
dotenv.load_dotenv('.env')

import gc
import os
import re
from peft import LoraConfig
from typing import List, Tuple, Dict, Any

from twinkle import get_logger, init_twinkle_client
from twinkle.reward.base import Reward
from twinkle.advantage import GRPOAdvantage
from twinkle.dataset import DatasetMeta, Dataset
from twinkle.metric import CompletionRewardMetric
from twinkle.dataloader import DataLoader
from twinkle.preprocessor.llm import GSM8KProcessor
from twinkle_client.model import MultiLoraTransformersModel
from twinkle_client.sampler import vLLMSampler

logger = get_logger()

# ========== 全局配置 ==========
MODEL_ID = 'ms://Qwen/Qwen3.6-35B-A3B'
NUM_GENERATIONS = 4
MAX_NEW_TOKENS = 1024
LEARNING_RATE = 2e-5
MAX_STEPS = 100
BATCH_SIZE = 2
TEMPERATURE = 1.0
SYNC_INTERVAL = 1
GRADIENT_ACCUMULATION_STEPS = 1
DATA_NUM = 2000

SYSTEM_PROMPT = (
    'You are a helpful math assistant. Solve the problem with minimal but correct reasoning '
    'and put your final answer within \\boxed{}.'
)


/root/miniconda3/envs/vllm19/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 第二步：定义奖励函数

GRPO 需要奖励函数来评判每条回答的质量。本例使用两个奖励函数：

### 准确性奖励 (MathAccuracyReward)
- 从模型输出中提取 `#### 数字` 格式的答案
- 与标准答案做数值比较
- 正确得 1.0 分，错误得 0.0 分

### 格式奖励 (MathFormatReward)
- 检查输出是否包含 `<step>...</step>` 推理标签和 `####` 答案标记
- 格式正确时，回答越短得分越高（鼓励简洁推理）
- 格式不正确得 0.0 分

**总奖励 = 准确性奖励 + 格式奖励**，最高 2.0 分。

In [ ]:
class MathAccuracyReward(Reward):
    """准确性奖励：检查模型答案是否与标准答案一致"""

    @staticmethod
    def extract_answer(completion: str) -> str:
        text = completion[-500:] if len(completion) > 500 else completion
        matches = re.findall(r'####\s*([\-\d,\.\s]+)', text)
        if matches:
            return matches[-1].replace(',', '').replace(' ', '').strip()
        return ''

    def __call__(self, trajectories: List[Dict[str, Any]], ground_truths: List[Dict[str, Any]]) -> List[float]:
        rewards = []
        for trajectory in trajectories:
            messages = trajectory.get('messages', [])
            completion = ''
            for msg in reversed(messages):
                if msg.get('role') == 'assistant':
                    completion = msg.get('content', '')
                    break

            gt = ''
            user_data = trajectory.get('user_data', [])
            if isinstance(user_data, list):
                for item in user_data:
                    if isinstance(item, (list, tuple)) and len(item) == 2:
                        if item[0] == 'ground_truth':
                            gt = str(item[1])
                            break

            predicted = self.extract_answer(completion)
            correct = False
            if predicted and gt:
                try:
                    correct = abs(float(predicted) - float(gt)) < 1e-5
                except (ValueError, OverflowError):
                    correct = predicted == gt

            rewards.append(1.0 if correct else 0.0)
        return rewards


class MathFormatReward(Reward):
    """格式奖励：检查格式并奖励简短回答"""

    def __call__(self, trajectories: List[Dict[str, Any]], ground_truths: List[Dict[str, Any]]) -> List[float]:
        rewards = []
        for trajectory in trajectories:
            messages = trajectory.get('messages', [])
            completion = ''
            for msg in reversed(messages):
                if msg.get('role') == 'assistant':
                    completion = msg.get('content', '')
                    break

            has_think = bool(re.search(r'<step>.*?</step>', completion, re.DOTALL))
            has_answer = bool(re.search(r'####\s*[\-\d,\.]+', completion))

            if not (has_think and has_answer):
                rewards.append(0.0)
            else:
                length = len(completion)
                if length <= 100:
                    rewards.append(1.0)
                else:
                    reward = max(0.0, 1.0 - (length - 100) / 2000)
                    rewards.append(reward)
        return rewards


def compute_rewards(trajectories: List[Dict[str, Any]]) -> Tuple[List[float], List[float], List[float]]:
    """计算总奖励 = 准确性 + 格式"""
    accuracy_reward_fn = MathAccuracyReward()
    format_reward_fn = MathFormatReward()
    accuracy_rewards = accuracy_reward_fn(trajectories, [])
    format_rewards = format_reward_fn(trajectories, [])
    total_rewards = [a + f for a, f in zip(accuracy_rewards, format_rewards)]
    return total_rewards, format_rewards, accuracy_rewards


## 第三步：准备数据集

加载 ModelScope 上的 `gsm8k` 数学数据集，并进行预处理和编码。

- `GSM8KProcessor`：提取题目和标准答案（`####` 格式），构造 system + user 对话
- `add_generation_prompt=True`：编码时在末尾加上 assistant 前缀，准备让模型生成回答
- `truncation_strategy='delete'`：超过最大长度的样本直接删除而非截断

In [ ]:
def create_math_dataset():
    meta = DatasetMeta(
        'ms://modelscope/gsm8k',
        subset_name='main',
        split='train',
        data_slice=range(DATA_NUM),
    )
    dataset = Dataset(meta)
    dataset.set_template('Qwen3_5Template', model_id=MODEL_ID, max_length=2048, enable_thinking=False)
    dataset.map(GSM8KProcessor(system=SYSTEM_PROMPT))
    dataset.encode(add_generation_prompt=True, truncation_strategy='delete')
    return dataset

dataset = create_math_dataset()
dataloader = DataLoader(dataset=dataset, batch_size=BATCH_SIZE, num_workers=0)
print(f'数据集加载完成，共 {len(dataset)} 条')

## 第四步：初始化 Twinkle 客户端与模型

Twinkle 客户端直接与训练服务通信，支持完整的模型配置：

- **`MultiLoraTransformersModel`**：支持 LoRA 适配器、损失函数、优化器、模板等全部设置
- **`vLLMSampler`**：采样端，支持 `adapter_uri` 动态加载最新 LoRA 权重
- **`LoraConfig`**：可以精确控制 `target_modules`，这是使用 twinkle 语法的关键优势

> 对于 MoE 模型，必须显式指定 `target_modules` 而非 `all-linear`，以避免 vLLM 兼容性问题。


In [ ]:
# 初始化 Twinkle 客户端
client = init_twinkle_client(
    base_url='http://www.modelscope.cn/twinkle',
    api_key=os.environ.get('MODELSCOPE_TOKEN', 'EMPTY_TOKEN'),
)

# 配置训练模型
model = MultiLoraTransformersModel(model_id=MODEL_ID)

# LoRA 配置 —— 显式指定 target_modules（MoE 模型关键）
lora_config = LoraConfig(
    target_modules=[
        'mlp.linear_fc1', 'mlp.linear_fc2',
        'attn.proj',
        'shared_experts.linear_fc1', 'shared_experts.linear_fc2',
        'linear_qkv', 'in_proj', 'out_proj', 'linear_proj',
    ],
    r=8,
    lora_alpha=32,
    lora_dropout=0.05,
)
model.add_adapter_to_model(
    'default',
    lora_config,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
)

# 设置 GRPO 损失函数（RL 训练的核心）
model.set_loss('GRPOLoss', epsilon=0.2, beta=0.0)

# 设置优化器
model.set_optimizer('Adam', lr=LEARNING_RATE)

# 设置输入处理器和模板
model.set_processor('InputProcessor')
model.set_template('Qwen3_5Template', model_id=MODEL_ID)

# 配置采样端
sampler = vLLMSampler(model_id=MODEL_ID)
sampler.set_template('Qwen3_5Template', model_id=MODEL_ID)

# 设置指标和优势函数
advantage_fn = GRPOAdvantage()
metrics = CompletionRewardMetric()

sampling_params = {
    'max_tokens': MAX_NEW_TOKENS,
    'temperature': TEMPERATURE,
    'top_p': 0.95,
    'num_samples': NUM_GENERATIONS,
    'logprobs': 1,
}

print('模型和采样端配置完成')

## 第五步：GRPO 训练主循环

每个训练步骤包含以下阶段：

### 5.1 保存权重
每隔 `SYNC_INTERVAL` 步调用 `model.save()` 保存 LoRA 权重，获取 `twinkle_path` 用于采样端加载。

### 5.2 采样生成
通过 `sampler.sample(inputs, adapter_uri=twinkle_path)` 使用最新 LoRA 权重生成回答。

### 5.3 计算奖励与优势
- 对每条回答计算准确性和格式奖励
- 用 `GRPOAdvantage` 在同组内标准化，得到优势值

### 5.4 训练步
调用 `model.forward_backward(inputs, advantages, old_logps)` 执行 GRPO 策略优化，然后 `model.clip_grad_and_step()` 更新参数。

> **与 tinker 语法的区别**：twinkle 客户端将 GRPO 的 forward_backward 和 optimizer step 封装为高级 API，无需手动构造 `Datum` 和 `loss_fn_inputs`。


In [ ]:
current_adapter_uri = None
step = 0

for batch in dataloader:
    if step >= MAX_STEPS:
        break

    metrics.reset()
    prompts = batch if isinstance(batch, list) else [batch]

    # ===== 5.1 保存权重并更新 adapter_uri =====
    if step % SYNC_INTERVAL == 0:
        logger.info(f'Step {step}: Saving weights for sampler...')
        result = model.save(
            name=f'grpo-sampler-step-{step}',
            save_optimizer=False,
        )
        current_adapter_uri = result.twinkle_path
        logger.info(f'Step {step}: Saved weights to {current_adapter_uri}')

    # ===== 5.2 采样生成 =====
    sample_responses = sampler.sample(
        inputs=prompts,
        sampling_params=sampling_params,
        adapter_uri=current_adapter_uri,
    )

    all_input_data: List[Dict[str, Any]] = []
    all_old_logps: List[List[float]] = []
    all_completion_lengths: List[int] = []

    for sample_response in sample_responses:
        for sequence in sample_response.sequences:
            all_input_data.append(sequence.new_input_feature)
            all_old_logps.append([logprob[0][1] for logprob in sequence.logprobs])
            all_completion_lengths.append(len(sequence.tokens))

    # ===== 5.3 计算奖励 =====
    total_rewards, format_rewards, accuracy_rewards = compute_rewards(all_input_data)
    metrics.accumulate(
        completion_lengths=all_completion_lengths,
        rewards={
            'total': total_rewards,
            'format': format_rewards,
            'accuracy': accuracy_rewards,
        },
    )

    # ===== 5.4 计算优势值 =====
    advantages = advantage_fn(
        total_rewards,
        num_generations=NUM_GENERATIONS,
        scale='group',
    ).tolist()

    frac_zero_std = (1.0 if all(abs(a) < 1e-8 for a in advantages) else 0.0)
    if frac_zero_std == 1.0:
        logger.info(f'Step {step}: All advantages are zero, skipping training')
        step += 1
        continue

    # ===== 5.5 GRPO 训练步 =====
    model.forward_backward(
        inputs=all_input_data,
        advantages=advantages,
        old_logps=all_old_logps,
    )
    model.clip_grad_and_step()

    gc.collect()

    # ===== 5.6 日志 =====
    log_dict = metrics.calculate()
    log_dict.update(model.calculate_metric(is_training=True).result)
    log_dict['train/frac_reward_zero_std'] = frac_zero_std
    logger.info(f'Step {step}: {log_dict}')
    step += 1


## 第六步：保存最终检查点

训练完成后保存最终的 LoRA 检查点，可用于后续推理（参见 `sample.ipynb`）。


In [ ]:
result = model.save(name='grpo-math-final', save_optimizer=True)
logger.info(f'Saved final checkpoint: {result}')
print(f'训练完成！检查点路径: {result.twinkle_path}')

## 上传到 ModelScope Hub（可选）

将训练好的 LoRA 检查点上传到 [ModelScope Hub](https://www.modelscope.cn)，方便共享、部署和后续合并权重。

| 参数 | 说明 |
|------|------|
| `checkpoint_dir` | 训练保存的检查点路径（`twinkle://...` 格式） |
| `hub_model_id` | 上传到 Hub 的模型 ID，格式为 `用户名/模型名` |
| `async_upload` | 是否异步上传，`False` 表示等待上传完成 |

In [ ]:
# （可选）上传到 ModelScope Hub
# YOUR_USER_NAME = 'your_username'
# hub_model_id = f'{YOUR_USER_NAME}/twinkle-grpo-math-lora'
# model.upload_to_hub(
#     checkpoint_dir=result.twinkle_path,
#     hub_model_id=hub_model_id,
#     async_upload=False,
# )
# print(f'检查点已上传至 Hub: {hub_model_id}')

## 推理（Inference）

训练完成后，可以直接使用 **线上服务** 进行推理，无需本地 GPU。

通过 `save_weights_and_get_sampling_client` 或 `create_sampling_client` 加载训练好的 LoRA 检查点，即可在线采样生成。

> 将下方 `weight_path` 替换为训练输出的检查点路径（`twinkle://...` 格式）。


> ⚠️ **MoE 模型在线推理限制**：由于 vLLM 目前不支持 Expert LoRA，在线推理无法直接加载 MoE 模型的 LoRA 权重。请先使用上方的「上传到 ModelScope Hub」功能将检查点上传，再通过下方的「合并权重并导出」得到完整模型后部署使用。

In [ ]:
# 推理示例（使用线上服务，无需本地 GPU）
import os
from tinker import types
from twinkle import init_tinker_client, get_logger
from twinkle.data_format import Message, Trajectory
from twinkle.template import Template

logger = get_logger()

BASE_MODEL = 'Qwen/Qwen3.6-35B-A3B'

# TODO: 替换为训练输出的检查点路径
weight_path = '<替换为你的 twinkle:// 检查点路径>'  # 例如: save_result.path

init_tinker_client()
from tinker import ServiceClient

service_client = ServiceClient(
    base_url='http://www.modelscope.cn/twinkle',
    api_key=os.environ.get('MODELSCOPE_TOKEN'),
)

# 加载 LoRA 检查点并创建采样客户端
sampling_client = service_client.create_sampling_client(
    model_path=weight_path,
    base_model=BASE_MODEL,
)

# 构造 Prompt
template = Template(model_id=f'ms://{BASE_MODEL}')
trajectory = Trajectory(
    messages=[
        Message(role='system', content='You are a helpful assistant.'),
        Message(role='user', content='你好，请介绍一下你自己。'),
    ]
)

input_feature = template.encode(trajectory, add_generation_prompt=True)
input_ids = input_feature['input_ids'].tolist()

# 采样
prompt = types.ModelInput.from_ints(input_ids)
params = types.SamplingParams(max_tokens=256, temperature=0.7)

print('Sampling...')
future = sampling_client.sample(prompt=prompt, sampling_params=params, num_samples=3)
result = future.result()

# 输出结果
print('Responses:')
for i, seq in enumerate(result.sequences):
    print(f'{i}: {repr(template.decode(seq.tokens))}')

## 合并权重并导出

训练得到的 LoRA 权重可以与原始模型合并，导出为完整的 HuggingFace 模型，方便后续部署和推理。

> **注意**：合并操作需要 GPU 资源（需要加载完整模型），请在有足够显存的环境下执行。

```bash
CUDA_VISIBLE_DEVICES=0,1,2,3 \
NPROC_PER_NODE=4 \
/opt/conda/envs/twinkle/bin/megatron export \
    --model Qwen/Qwen3.6-35B-A3B \
    --adapters <替换为你的 LoRA 检查点路径> \
    --output_dir <替换为输出目录> \
    --merge_lora true \
    --to_hf true \
    --tensor_model_parallel_size 2 \
    --expert_model_parallel_size 2 \
    --pipeline_model_parallel_size 2
```

**参数说明**：

| 参数 | 说明 |
|------|------|
| `--model` | 基座模型 ID |
| `--adapters` | 训练保存的 LoRA 检查点路径 |
| `--output_dir` | 合并后的完整模型输出目录 |
| `--merge_lora true` | 将 LoRA 权重合并到基座模型中 |
| `--to_hf true` | 导出为 HuggingFace 格式 |
| `--tensor_model_parallel_size` | 张量并行大小 |
| `--expert_model_parallel_size` | 专家并行大小（MoE 模型专用） |
| `--pipeline_model_parallel_size` | 流水线并行大小 |

合并完成后，输出目录中即为完整的 HuggingFace 模型，可直接用于推理或部署。

## 关键指标解读

训练过程中会输出以下指标：

| 指标 | 含义 | 期望趋势 |
|------|------|----------|
| `accuracy` | 回答正确率 | 逐步上升 |
| `format` | 格式正确率 | 快速达到高值 |
| `total` | 总奖励（准确性+格式） | 逐步上升 |
| `frac_reward_zero_std` | 同组奖励标准差为零的比例 | 逐步下降（说明模型在区分好坏回答） |
| `completion_lengths` | 回答平均长度 | 逐步缩短（简洁奖励的效果） |

## 常见问题

| 问题 | 可能原因 | 解决方法 |
|------|----------|----------|
| 准确率不提升 | 学习率太低/太高 | 尝试 5e-5 或 2e-4 |
| 所有 advantage 为 0 | 同组回答奖励完全相同 | 增大 NUM_GENERATIONS 或提高 temperature |
| OOM 内存不足 | 生成太长 | 减小 MAX_NEW_TOKENS 或 BATCH_SIZE |
| 采样超时 | 服务端 sampler 未配置 | 检查 server_config.yaml 中 sampler 配置 |